# Integración de fuentes externas: agroclima, suelo y ERA5-Land (03_externas)

Este notebook incorpora al proyecto las variables ambientales que no provienen de las imágenes de las parcelas: **lluvia diaria**, **propiedades del suelo** y **variables meteorológicas de reanálisis**. Todas se extraen con **Google Earth Engine**, una plataforma que aloja catálogos públicos de datos geoespaciales y ejecuta en sus servidores los cálculos de reducción espacial (Gorelick et al., 2017). Así no es necesario descargar los ráster completos: solo se transfiere una tabla con el valor de cada parcela.

Las variables se calculan **dentro de las etapas fenológicas de cada parcela** que se estimaron en (`fenologia_parcela.parquet`). Dos parcelas del mismo píxel climático reciben valores distintos si su ciclo empezó en fechas distintas.


**Fuentes externas y procedencia**

| Fuente | Producto | Acceso (Earth Engine) | Resolución nativa | Periodo descargado | Cita |
|---|---|---|---|---|---|
| Lluvia | CHIRPS v2.0 diario | `UCSB-CHG/CHIRPS/DAILY`, banda `precipitation` | 0.05° (≈ 5.5 km) | 2022-01-01 a 2025-12-31 | Funk et al. (2015) |
| Suelo | SoilGrids 2.0 (media, 0–30 cm) | `projects/soilgrids-isric/{clay, sand, soc, bdod, phh2o}_mean` | 250 m | Estático | Poggio et al. (2021) |
| Meteorología | ERA5-Land, agregados diarios | `ECMWF/ERA5_LAND/DAILY_AGGR` | 0.1° (≈ 9–11 km) | 2022-01-01 a 2025-12-31 | Muñoz-Sabater et al. (2021) |
| Temperatura (insumo del Paso 3) | Tmax/Tmin mensual corregida por elevación | `clima_mensual_parcela.parquet` (local) | Malla de clima 0.05° | 2022–2025 | Funk et al. (2019), para CHIRTS |

**Nota de procedencia**: la columna de temperatura corregida (`tmin_c_ajust` y `tmax_c_ajust`) no se calcula aquí, sino en el primer archivo. El diccionario del maestro la describe como "CHIRTS corregido por elevación". Su método se docupenta en ese notebook.

In [ ]:
import ee
#autenticacion para poder tener acceso a la API de Google Earth Engine 
ee.Authenticate()

True

## Conexión con Earth Engine y prueba de extracción
**Problemas**: las 197 parcelas son polígonos de pocas hectáreas, mientras que los píxeles de CHIRPS y de ERA5-Land miden varios kilómetros. Si se pide la media zonal a la resolución nativa, una parcela pequeña puede ocupar una fracción mínima de un píxel y quedarse **sin valor** (`null`).

**lo que se hizo:**
- Las parcelas se suben a Earth Engine como una `FeatureCollection` en WGS84 que contiene solo la clave y la geometría. Se verifica que lleguen las 197.
- La media por parcela se extrae c on `reduceRegions` y `ee.Reducer.mean()`, a una **escala mucho más fina que la nativa**: $250$ m para CHIRPS, $50$ m para SoilGrids y $1{,}000$ m para ERA5-Land.
- Antes de la descarga completa se prueba **un solo día** (15 de julio de 2025) y se compara **julio de 2025** contra el archivo mensual del Paso 3, que proviene de la misma fuente.

**Justificación**
- En Earth Engine, un reductor ponderado como la media incluye un píxel cuando al menos $\approx 1/256$ de su área cae dentro de la región. Su peso es la fracción cubierta, y si la región es muy pequeña frente al píxel, los pesos se vuelven cero y el resultado es `null`. La documentación recomienda fijar siempre la escala y usar una más pequeña en esos casos (Google, s. f.-b).
- Al pedir una escala distinta de la nativa, Earth Engine **remuestrea por vecino más cercano** (Google, s. f.-a). El valor de cada parcela sigue siendo el de los píxeles originales que la cubren, ponderado por área, no un valor interpolado.

In [ ]:
# se tiene Conexión con Earth Engine y prueba de extracción por parcela.
from __future__ import annotations
import json
import logging
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
logger = logging.getLogger("agrocebada.agroclima")
CLAVE: str = "id_poligono"
# Raíz del repositorio: se sube desde la carpeta actual hasta encontrar main.py
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "main.py").exists())
DIR_INTERIM: Path = RAIZ / "data" / "interim"
PROYECTO_EE: str = "tu-proyecto-de-earth-engine"   # cambiar por el ID de un proyecto propio de Earth Engine
try:
    ee.Initialize(project=PROYECTO_EE)
except ee.EEException:
    ee.Authenticate(auth_mode="localhost")
    ee.Initialize(project=PROYECTO_EE)
print("Earth Engine listo · proyecto:", PROYECTO_EE)
parcelas = gpd.read_parquet(DIR_INTERIM / "parcelas_base.parquet")
fenologia = pd.read_parquet(DIR_INTERIM / "fenologia_parcela.parquet")
# Las 197 parcelas como FeatureCollection de Earth Engine (solo clave y geometría).
geojson = json.loads(parcelas[[CLAVE, "geometry"]].to_crs(4326).to_json())
parcelas_ee = ee.FeatureCollection(geojson)
assert parcelas_ee.size().getInfo() == 197, "No se subieron las 197 parcelas"
def extraer_por_parcela(imagen: ee.Image, escala_m: float) -> pd.DataFrame:
    """Media de cada banda por parcela.
    La escala se fija MUY por debajo de la nativa para que Earth Engine remuestree
    por vecino más cercano: así ninguna parcela pequeña queda sin valor (null).
    """
    resultado = imagen.reduceRegions(
        collection=parcelas_ee, reducer=ee.Reducer.mean(), scale=escala_m, tileScale=4
    ).getInfo()
    filas = [f["properties"] for f in resultado["features"]]
    return pd.DataFrame(filas)
# Prueba: lluvia de un solo día (15 de julio de 2025).
dia = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate("2025-07-15", "2025-07-16").first()
prueba = extraer_por_parcela(dia.select("precipitation"), escala_m=250)
assert len(prueba) == 197 and prueba[CLAVE].is_unique
print("Parcelas sin valor:", int(prueba["mean"].isna().sum()))
print(prueba["mean"].describe().round(2))
# Control de coherencia con lo que ya tienes: julio de 2025 sumado debe parecerse
# a tu prec_mm mensual del Paso 3 (misma fuente, misma malla).
julio = (ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
         .filterDate("2025-07-01", "2025-08-01").select("precipitation").sum())
julio_ee = extraer_por_parcela(julio, escala_m=250).rename(columns={"mean": "prec_julio_ee"})
clima = pd.read_parquet(DIR_INTERIM / "clima_mensual_parcela.parquet")
julio_local = clima.query("anio == 2025 and mes == 7")[[CLAVE, "prec_mm"]]
control = julio_ee.merge(julio_local, on=CLAVE, validate="one_to_one")
dif = control["prec_julio_ee"] - control["prec_mm"]
print(f"\nJulio 2025 · Earth Engine − archivo mensual (mm): "
      f"mediana {dif.median():.1f} | máx. abs. {dif.abs().max():.1f}")

Earth Engine listo · proyecto: tu-proyecto-de-earth-engine
Parcelas sin valor: 0
count    197.00
mean       7.76
std        2.24
min        5.20
25%        6.62
50%        6.99
75%        7.93
max       15.53
Name: mean, dtype: float64

Julio 2025 · Earth Engine − archivo mensual (mm): mediana -0.0 | máx. abs. 16.6


## Lluvia diaria CHIRPS (2022–2025)

**Fuente**: CHIRPS (*Climate Hazards Group InfraRed Precipitation with Station data*) combina estimaciones de precipitación por infrarrojo térmico con datos de estaciones, en una malla de $0.05°$ con cobertura casi global desde 1981 (Funk et al., 2015). En México, una evaluación contra estaciones de CONAGUA en cuencas de San Luis Potosí y del Estado de México encontró correlaciones mensuales significativas ($\rho \approx 0.73$–$0.98$), con subestimación sistemática en la cuenca tropical. Los autores lo consideran un complemento útil donde hay pocas estaciones (Rodríguez-Herrera et al., 2025).

**Decisiones**
1. **Periodo de 2022 a 2025**: 2025 es el ciclo productivo y 2022–2024 sirven como **años base** para calcular anomalías
2. **Descarga mensual con caché**: cada mes se pide en **una sola llamada** (`toBands()` convierte los días del mes en bandas) y se guarda en `chirps_diario/AAAA_MM.parquet`
3. **Contratos**: cada mes debe tener tantas imágenes como días. La tabla final debe tener $197 \times 1{,}461$ filas, sin duplicados, sin faltantes y sin lluvia negativa.


**Justificación**: la caché por mes hace el proceso **reanudable e idempotente**: si la conexión falla en el mes 30, los 29 anteriores no se vuelven a descargar, y una segunda ejecución produce el mismo resultado. Contar las imágenes por mes detecta días faltantes en el catálogo antes de agregarlos. Los contratos aplican la idea de verificar los datos como si fueran código (Schelter et al., 2018).

In [ ]:
 #  CHIRPS diario 2022–2025 por parcela (Earth Engine). Se descarga en caché mes a mes, luego se concatena todo.

COLECCION_CHIRPS: str = "UCSB-CHG/CHIRPS/DAILY"
ESCALA_CHIRPS_M: float = 250.0
DIR_CACHE_CHIRPS: Path = DIR_INTERIM / "chirps_diario"
DIR_CACHE_CHIRPS.mkdir(parents=True, exist_ok=True)
MESES: pd.PeriodIndex = pd.period_range("2022-01", "2025-12", freq="M")
def descargar_chirps_mes(periodo: pd.Period, reintentos: int = 3):
    """Descarga la colección CHIRPS diaria de un mes y la reduce por parcela.
        periodo: Mes a descargar.
        reintentos: Intentos ante errores transitorios del servidor."""
    inicio = periodo.start_time.strftime("%Y-%m-%d")
    fin = (periodo + 1).start_time.strftime("%Y-%m-%d")
    coleccion = (ee.ImageCollection(COLECCION_CHIRPS)
                 .filterDate(inicio, fin).select("precipitation"))

    n_imagenes = coleccion.size().getInfo()
    if n_imagenes != periodo.days_in_month:
        raise ValueError(f"{periodo}: {n_imagenes} imágenes; se esperaban {periodo.days_in_month}")

    # Cada día es una banda: '20250701_precipitation', '20250702_precipitation', ...
    imagen_mes = coleccion.toBands()
    for intento in range(1, reintentos + 1):
        try:
            resultado = imagen_mes.reduceRegions(
                collection=parcelas_ee, reducer=ee.Reducer.mean(),
                scale=ESCALA_CHIRPS_M, tileScale=4,
            ).getInfo()
            break
        except ee.EEException as error:
            if intento == reintentos:
                raise
            logger.warning("%s: intento %d falló (%s); reintentando…", periodo, intento, error)
            time.sleep(10 * intento)

    ancho = pd.DataFrame([f["properties"] for f in resultado["features"]])
    largo = ancho.melt(id_vars=CLAVE, var_name="banda", value_name="prec_mm")
    largo["fecha"] = pd.to_datetime(largo["banda"].str[:8], format="%Y%m%d")
    return largo[[CLAVE, "fecha", "prec_mm"]]


# Descarga con caché: si un mes ya está en disco, no se vuelve a pedir.
inicio_reloj = time.perf_counter()
for periodo in MESES:
    ruta_mes = DIR_CACHE_CHIRPS / f"{periodo.strftime('%Y_%m')}.parquet"
    if ruta_mes.exists():
        continue
    descargar_chirps_mes(periodo).to_parquet(ruta_mes, index=False)
    logger.info("CHIRPS %s descargado", periodo)
logger.info("Descarga terminada en %.1f min", (time.perf_counter() - inicio_reloj) / 60)

chirps_diario = (
    pd.concat([pd.read_parquet(r) for r in sorted(DIR_CACHE_CHIRPS.glob("*.parquet"))],
              ignore_index=True)
    .sort_values([CLAVE, "fecha"]).reset_index(drop=True)
)

# Contratos.
n_dias = len(pd.date_range("2022-01-01", "2025-12-31", freq="D"))      # 1,461 (2024 es bisiesto)
assert len(chirps_diario) == 197 * n_dias, f"Filas: {len(chirps_diario)} (esperadas {197 * n_dias})"
assert not chirps_diario.duplicated([CLAVE, "fecha"]).any(), "Días duplicados"
assert chirps_diario["prec_mm"].notna().all(), "Días sin valor"
assert (chirps_diario["prec_mm"] >= 0).all(), "Lluvia negativa"

# Control en los 48 meses contra el archivo mensual del Paso 3 (misma fuente).
clima = pd.read_parquet(DIR_INTERIM / "clima_mensual_parcela.parquet")
suma_mes = (chirps_diario.assign(anio=chirps_diario["fecha"].dt.year, mes=chirps_diario["fecha"].dt.month)
            .groupby([CLAVE, "anio", "mes"], as_index=False)["prec_mm"].sum()
            .rename(columns={"prec_mm": "prec_mm_diario"}))
control = suma_mes.merge(clima[[CLAVE, "anio", "mes", "prec_mm"]],
                         on=[CLAVE, "anio", "mes"], validate="one_to_one")
dif = control["prec_mm_diario"] - control["prec_mm"]
print(f"Diario sumado − mensual (mm) · mediana {dif.median():.2f} | "
      f"p95 abs {dif.abs().quantile(0.95):.1f} | máx. abs {dif.abs().max():.1f}")
print("Correlación diario-sumado vs mensual:", round(control[["prec_mm_diario", "prec_mm"]].corr().iloc[0, 1], 4))

chirps_diario.to_parquet(DIR_INTERIM / "chirps_diario_parcela.parquet", index=False)
print(f"Guardado chirps_diario_parcela: {chirps_diario.shape}")

INFO | CHIRPS 2022-01 descargado
INFO | CHIRPS 2022-02 descargado
INFO | CHIRPS 2022-03 descargado
INFO | CHIRPS 2022-04 descargado
INFO | CHIRPS 2022-05 descargado
INFO | CHIRPS 2022-06 descargado
INFO | CHIRPS 2022-07 descargado
INFO | CHIRPS 2022-08 descargado
INFO | CHIRPS 2022-09 descargado
INFO | CHIRPS 2022-10 descargado
INFO | CHIRPS 2022-11 descargado
INFO | CHIRPS 2022-12 descargado
INFO | CHIRPS 2023-01 descargado
INFO | CHIRPS 2023-02 descargado
INFO | CHIRPS 2023-03 descargado
INFO | CHIRPS 2023-04 descargado
INFO | CHIRPS 2023-05 descargado
INFO | CHIRPS 2023-06 descargado
INFO | CHIRPS 2023-07 descargado
INFO | CHIRPS 2023-08 descargado
INFO | CHIRPS 2023-09 descargado
INFO | CHIRPS 2023-10 descargado
INFO | CHIRPS 2023-11 descargado
INFO | CHIRPS 2023-12 descargado
INFO | CHIRPS 2024-01 descargado
INFO | CHIRPS 2024-02 descargado
INFO | CHIRPS 2024-03 descargado
INFO | CHIRPS 2024-04 descargado
INFO | CHIRPS 2024-05 descargado
INFO | CHIRPS 2024-06 descargado
INFO | CHI

Diario sumado − mensual (mm) · mediana 0.00 | p95 abs 3.7 | máx. abs 38.2
Correlación diario-sumado vs mensual: 0.9991
Guardado chirps_diario_parcela: (287817, 3)


## Resolución efectiva de la lluvia por parcela

si varias parcelas caen en el mismo píxel de CHIRPS, ¿sus series de lluvia son prácticamente idénticas? Y si es así, ¿de dónde viene la variación entre parcelas que tendrán las variables agroclimáticas?

Por ellos se calcula la correlación entre la serie diaria de cada parcela en 2025 y la serie media de su píxel de clima (`clima_pixel_id`).
Ademas se calcula la **fracción de la varianza que ocurre dentro de los píxeles**: la varianza de los residuos respecto a la media del píxel, dividida entre la varianza total. 
Finalmente se compara esa fracción para la lluvia acumulada en una **ventana fija** (junio a septiembre) y en la **ventana fenológica** de cada parcela (`sos50` a `eos50`).


In [ ]:
# ¿Se conserva el píxel? ¿Cuánta variación aporta la ventana fenológica?
topo = pd.read_parquet(DIR_INTERIM / "topo_parcela.parquet", columns=[CLAVE, "clima_pixel_id"])
pixel = topo.set_index(CLAVE)["clima_pixel_id"]

# Identificación: cada parcela debe parecerse a la serie media de su píxel.
serie_2025 = (chirps_diario[chirps_diario["fecha"].dt.year == 2025]
              .pivot(index="fecha", columns=CLAVE, values="prec_mm"))
media_pixel = serie_2025.T.groupby(pixel).mean().T                 # una serie por píxel
corr_con_pixel = pd.Series({
    pid: serie_2025[pid].corr(media_pixel[pixel[pid]]) for pid in serie_2025.columns})
print("Correlación de cada parcela con la serie media de su píxel:")
print(corr_con_pixel.describe(percentiles=[0.05, 0.5]).round(3).to_string())
print("Parcelas de borde (corr < 0.95):", corr_con_pixel[corr_con_pixel < 0.95].index.tolist(), "\n")


# ¿Qué fracción de la varianza ocurre DENTRO de los píxeles?
def fraccion_dentro_pixel(valores: pd.Series) -> float:
    """Varianza dentro de los píxeles / varianza total (0 = todo lo explica el píxel)."""
    grupo = pixel.reindex(valores.index)
    dentro = valores - valores.groupby(grupo).transform("mean")
    total = valores.var()
    return float(dentro.var() / total) if total > 0 else float("nan")   # constante: no aplica


d = chirps_diario[chirps_diario["fecha"].dt.year == 2025].merge(
    fenologia[[CLAVE, "pl_sos50_doy", "pl_eos50_doy"]], on=CLAVE, validate="many_to_one")
doy = d["fecha"].dt.dayofyear
lluvia_fija = d[d["fecha"].between("2025-06-01", "2025-09-30")].groupby(CLAVE)["prec_mm"].sum()
lluvia_ciclo = d[doy.between(d["pl_sos50_doy"], d["pl_eos50_doy"])].groupby(CLAVE)["prec_mm"].sum()

print(f"Varianza dentro de los píxeles · ventana fija jun–sep: {fraccion_dentro_pixel(lluvia_fija):.0%}")
print(f"Varianza dentro de los píxeles · ventana fenológica:   {fraccion_dentro_pixel(lluvia_ciclo):.0%}")

Correlación de cada parcela con la serie media de su píxel:
count    197.000
mean       0.999
std        0.005
min        0.953
5%         0.998
50%        1.000
max        1.000
Parcelas de borde (corr < 0.95): [] 

Varianza dentro de los píxeles · ventana fija jun–sep: 1%
Varianza dentro de los píxeles · ventana fenológica:   46%



**Interpretaciones**

- **Identificación del píxel**: la correlación de cada parcela con la serie de su píxel tiene una mediana de $1.000$ y un mínimo de $0.953$. Ninguna parcela queda por debajo de $0.95$. Lo que significa que la extracción se asigna correctamente cada parcela 
- **Ventana fija**: solo el $1\%$ de la varianza ocurre dentro de los píxeles. Con fechas iguales para todas, parcelas del mismo píxel reciben prácticamente la misma lluvia.
- **Ventana fenológica**: la fracción sube a $46\%$. **Casi la mitad de la variación entre parcelas viene de las fechas fenológicas de cada una, no de la lluvia en sí.**

**Consecuencias para el análisis EDA**
- Las variables `ac_` combinan dos señales: el clima del píxel y el calendario estimado con Planet.

## Variables agroclimáticas por etapa fenológica

**Problemas**: el efecto de la lluvia y la temperatura sobre el rendimiento depende de **cuándo** ocurren respecto al desarrollo del cultivo. La misma lluvia acumulada no tiene el mismo efecto durante la floración que antes de la siembra.

**Decisión tomada: crear ventanas por etapa en el tiempo**

| Etapa | Ventana | Supuesto |
|---|---|---|
| `pre` | 30 días antes de `sos20` | Humedad disponible antes del establecimiento |
| `veg` | De `sos20` a `centro − 16` | Desarrollo vegetativo |
| `flor` | `centro` $\pm$ 15 días | Floración, centrada en el máximo de verdor |
| `llen` | De `centro + 16` a **`eos20`** | Llenado de grano |
| `ciclo` | De `sos20` a `eos20` | Ciclo completo de verdor |

**Variables**: lluvia por etapa, días con lluvia, lluvia máxima diaria, racha seca máxima (en floración y en el ciclo), grados-día (GDD) del ciclo y hasta la floración, evapotranspiración de referencia (ETo), balance lluvia − ETo (ciclo, floración y llenado) y anomalía de la lluvia del ciclo frente a 2022–2024.

**Justificación de cada cálculo**
- **Para el dia seco**: se considera seco el día con menos de $1$ mm. Es el mismo umbral que usan los índices de extremos climáticos del ETCCDI para los días húmedos y las rachas secas (Zhang et al., 2011).
- **Grados-día**: $\text{GDD} = \max(T_{\text{media}} - T_{\text{base}},\ 0)$ con $T_{\text{base}} = 5\ °\text{C}$. McMaster y Wilhelm (1997) muestran que el método de cálculo y la temperatura base cambian sustancialmente el total acumulado, y piden reportar ambos de forma explícita. Aquí se usa el método de la temperatura media truncada en cero. La base de $5\ °\text{C}$ es un **criterio práctico**.
- **Radiación extraterrestre** $R_a$: ecuaciones 21 a 25 de FAO-56 (Allen et al., 1998), con la latitud del punto representativo de cada parcela.
- **ETo de Hargreaves-Samani**:

$$ \text{ETo} = 0.0023 \cdot 0.408\,R_a \,(T_{\text{media}} + 17.8)\sqrt{T_{\text{max}} - T_{\text{min}}} $$

  Es la ecuación 52 de FAO-56. Se recomienda cuando solo se dispone de temperatura (Hargreaves y Samani, 1985; Allen et al., 1998).
- **Anomalía**: la lluvia de 2025 en la **misma ventana de días del año** de cada parcela se compara contra la media de 2022–2024. Al usar la misma ventana, la anomalía no se confunde con la duración del ciclo.

**Alternativas existentes**: la ETo de Penman-Monteith de FAO-56 es el método de referencia (Allen et al., 1998). Necesita radiación, humedad y viento, que ERA5-Land ofrece, pero no se usó aquí.

In [ ]:
#  Variables agroclimáticas por etapa fenológica de cada parcela (prefijo ac_).
TBASE_GDD: float = 5.0
UMBRAL_LLUVIA_MM: float = 1.0 # un día con < 1 mm cuenta como seco
MEDIA_VENTANA_FLOR: int = 15
DIAS_PRE_SIEMBRA: int = 30
ANIO_CICLO: int = 2025
ANIOS_BASE: tuple[int, ...] = (2022, 2023, 2024)
#   taza diaria (MJ/m²/día), ecuación 21 de FAO-56
def radiacion_extraterrestre(lat_grados: np.ndarray, doy: np.ndarray):
    phi = np.deg2rad(lat_grados)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    return (24 * 60 / np.pi) * 0.0820 * dr * (
        ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))
#Mayor número de días consecutivos con lluvia < UMBRAL_LLUVIA_MM.
def racha_seca_max(lluvia: np.ndarray):
    seco = (lluvia < UMBRAL_LLUVIA_MM).astype(int)
    if not seco.any():
        return 0
    cambios = np.diff(np.r_[0, seco, 0])
    return int((np.flatnonzero(cambios == -1) - np.flatnonzero(cambios == 1)).max())


# Tabla diaria completa: lluvia + temperatura corregida + Ra + GDD + ETo.
latitud = pd.DataFrame({
    CLAVE: parcelas[CLAVE].to_numpy(),
    "lat": parcelas.to_crs(4326).representative_point().y.to_numpy(),})
diario = (
    chirps_diario.assign(anio=chirps_diario["fecha"].dt.year,
                         mes=chirps_diario["fecha"].dt.month,
                         doy=chirps_diario["fecha"].dt.dayofyear)
    .merge(clima[[CLAVE, "anio", "mes", "tmin_c_ajust", "tmax_c_ajust"]],
           on=[CLAVE, "anio", "mes"], how="left", validate="many_to_one")
    .merge(latitud, on=CLAVE, how="left", validate="many_to_one"))
assert diario[["tmin_c_ajust", "tmax_c_ajust", "lat"]].notna().all().all(), "Días sin temperatura o latitud"

diario["tmedia"] = (diario["tmin_c_ajust"] + diario["tmax_c_ajust"]) / 2
diario["gdd"] = (diario["tmedia"] - TBASE_GDD).clip(lower=0)
diario["ra"] = radiacion_extraterrestre(diario["lat"].to_numpy(), diario["doy"].to_numpy())
diario["eto_mm"] = (0.0023 * 0.408 * diario["ra"] * (diario["tmedia"] + 17.8)
                    * np.sqrt((diario["tmax_c_ajust"] - diario["tmin_c_ajust"]).clip(lower=0)))


# Variables por parcela dentro de SUS etapas fenológicas.
def variables_parcela(g: pd.DataFrame, f: pd.Series) -> dict[str, float]:
    """Lluvia, rachas, GDD, ETo, balance y anomalía en las etapas de una parcela."""
    sos20, centro, eos50, eos20 = (int(f[c]) for c in
                                   ("pl_sos20_doy", "pl_centro_pico_doy", "pl_eos50_doy", "pl_eos20_doy"))
    ventanas = {
        "pre": (sos20 - DIAS_PRE_SIEMBRA, sos20 - 1),
        "veg": (sos20, centro - MEDIA_VENTANA_FLOR - 1),
        "flor": (centro - MEDIA_VENTANA_FLOR, centro + MEDIA_VENTANA_FLOR),
                "llen": (centro + MEDIA_VENTANA_FLOR + 1, eos20),   # eos20 ≥ eos50: la ventana no queda vacía
        "ciclo": (sos20, eos20),}
    g25 = g[g["anio"] == ANIO_CICLO]
    en = {e: g25[g25["doy"].between(a, b)] for e, (a, b) in ventanas.items()}

    fila: dict[str, float] = {f"ac_lluvia_{e}_mm": float(w["prec_mm"].sum()) for e, w in en.items()}
    fila |= {
        "ac_dias_lluvia_ciclo": int((en["ciclo"]["prec_mm"] >= UMBRAL_LLUVIA_MM).sum()),
        "ac_lluvia_max_dia_ciclo_mm": float(en["ciclo"]["prec_mm"].max()),
        "ac_racha_seca_max_flor": racha_seca_max(en["flor"]["prec_mm"].to_numpy()),
        "ac_racha_seca_max_ciclo": racha_seca_max(en["ciclo"]["prec_mm"].to_numpy()),
        "ac_gdd_ciclo": float(en["ciclo"]["gdd"].sum()),
        "ac_gdd_hasta_flor": float(g25.loc[g25["doy"].between(sos20, centro), "gdd"].sum()),
        "ac_eto_ciclo_mm": float(en["ciclo"]["eto_mm"].sum()),
        "ac_balance_ciclo_mm": float(en["ciclo"]["prec_mm"].sum() - en["ciclo"]["eto_mm"].sum()),
        "ac_balance_flor_mm": float(en["flor"]["prec_mm"].sum() - en["flor"]["eto_mm"].sum()),
        "ac_balance_llen_mm": float(en["llen"]["prec_mm"].sum() - en["llen"]["eto_mm"].sum()),}

    # Anomalía: la MISMA ventana de días del año de esta parcela, en los años base.
    base = np.mean([g.loc[(g["anio"] == a) & g["doy"].between(sos20, eos20), "prec_mm"].sum()
                    for a in ANIOS_BASE])
    fila["ac_lluvia_ciclo_anom_mm"] = fila["ac_lluvia_ciclo_mm"] - base
    fila["ac_lluvia_ciclo_anom_rel"] = fila["ac_lluvia_ciclo_mm"] / base - 1 if base > 0 else np.nan
    return fila


fen = fenologia.set_index(CLAVE)
agroclima = pd.DataFrame([
    {CLAVE: pid, **variables_parcela(g, fen.loc[pid])}
    for pid, g in diario.groupby(CLAVE, sort=True)
])

# Contratos.
assert len(agroclima) == 197 and agroclima[CLAVE].is_unique
assert agroclima.drop(columns=CLAVE).notna().all().all(), "Variables agroclimáticas con NaN"
assert (agroclima.filter(like="ac_lluvia_").drop(columns=["ac_lluvia_ciclo_anom_mm",
        "ac_lluvia_ciclo_anom_rel"]) >= 0).all().all(), "Lluvia negativa"
assert agroclima["ac_gdd_ciclo"].between(500, 3000).all(), "GDD del ciclo fuera de rango plausible"

print(agroclima.drop(columns=CLAVE).describe().round(1).T[["min", "25%", "50%", "75%", "max"]])

# ¿Cuánta variación ocurre DENTRO de los píxeles? (función de la 5.1b)
print("\nVarianza dentro de los píxeles de clima:")
for col in ("ac_lluvia_ciclo_mm", "ac_lluvia_flor_mm", "ac_racha_seca_max_flor",
            "ac_gdd_ciclo", "ac_balance_flor_mm", "ac_lluvia_ciclo_anom_rel"):
    print(f"  {col:28s} {fraccion_dentro_pixel(agroclima.set_index(CLAVE)[col]):5.0%}")

agroclima.to_parquet(DIR_INTERIM / "agroclima_parcela.parquet", index=False)
print(f"\nGuardado agroclima_parcela: {agroclima.shape}")

                               min     25%     50%     75%     max
ac_lluvia_pre_mm              48.1   117.1   155.9   223.0   381.0
ac_lluvia_veg_mm              28.3   272.5   362.2   434.0   693.3
ac_lluvia_flor_mm             75.8   116.9   146.4   177.7   331.6
ac_lluvia_llen_mm              0.0   130.9   185.6   276.1   663.9
ac_lluvia_ciclo_mm           274.7   633.4   696.2   783.7  1157.1
ac_dias_lluvia_ciclo          16.0    45.0    51.0    53.0    61.0
ac_lluvia_max_dia_ciclo_mm    25.0    36.1    42.0    45.6    92.7
ac_racha_seca_max_flor         4.0     5.0     6.0     9.0    19.0
ac_racha_seca_max_ciclo        9.0    25.0    30.0    33.0    41.0
ac_gdd_ciclo                1082.5  1748.1  1948.0  2075.5  2291.4
ac_gdd_hasta_flor            385.6   798.7  1004.3  1283.3  1710.3
ac_eto_ciclo_mm              309.3   543.5   582.3   610.9   670.6
ac_balance_ciclo_mm          -64.5    35.1   120.4   197.3   547.3
ac_balance_flor_mm           -50.6    -2.8    24.9    58.1   2

**Interpretaciones**

- **Lluvia del ciclo**: la mediana es de $696$ mm ($275$–$1{,}157$). La floración recibe una mediana de $146$ mm y hay $51$ días con lluvia en el ciclo.
- **Balance hídrico**: positivo en el ciclo (mediana $+120$ mm) y negativo en el llenado (mediana $-9.8$ mm). La demanda atmosférica supera a la lluvia hacia el final del temporal.
- **Racha seca del ciclo**: la mediana es de $30$ días, frente a $6$ en floración. Como el ciclo llega hasta `eos20`, incluye el fin de la temporada de lluvias.
- **Anomalía**: la lluvia del ciclo de 2025 fue mayor que la media de 2022–2024 en **todas** las parcelas (mínimo $+7.9$ mm, mediana $+120$ mm, $\approx 20\%$ más).
- **Contratos**: $197$ parcelas, sin faltantes, lluvia no negativa y GDD del ciclo dentro de $[500, 3000]$ ($1{,}083$–$2{,}291$).
- **Variación dentro de los píxeles**: va de $22\%$ (lluvia en floración) a $80\%$ (GDD del ciclo). El GDD varía tanto dentro del píxel porque depende de la duración del ciclo y de la temperatura corregida por elevación de cada parcela.

**Cuestiones a considerar, explorar y analizar**
- **Temperatura mensual aplicada a días**: `tmin_c_ajust` y `tmax_c_ajust` son mensuales, de modo que el GDD y la ETo diarios son constantes dentro de cada mes y se pierde la variabilidad de día a día.
- **Base climática corta**: tres años no forman una normal climatológica. La OMM define las normales sobre periodos de 30 años (World Meteorological Organization [WMO], 2017). Con esta base, la anomalía compara solo contra tres años recientes.
- **`pre` no es pre-siembra**: `sos20` marca el inicio del verdor detectado por satélite, que ocurre después de la siembra. La etapa `pre` incluye parte del periodo posterior a la siembra.
- **Ventanas por etapa**: los $\pm 15$ días de floración y los 30 días previos son **criterios prácticos** nuevamente, no fechas agronómicas observadas.


###  Control de ventanas vacías o demasiado cortas

**Objetivo**: verificar que ninguna parcela tenga una etapa vegetativa o de llenado de grano tan corta (7 días o menos) que sus variables se calculen con muy pocos días.

In [ ]:
# ¿Hay ventanas de etapa vacías o demasiado cortas?
f = fenologia.set_index(CLAVE)
largo_veg = (f["pl_centro_pico_doy"] - MEDIA_VENTANA_FLOR - 1) - f["pl_sos20_doy"] + 1
largo_llen = f["pl_eos50_doy"] - (f["pl_centro_pico_doy"] + MEDIA_VENTANA_FLOR + 1) + 1
print("Días de la etapa vegetativa:", largo_veg.describe().round(0).to_dict())
print("Días de la etapa de llenado:", largo_llen.describe().round(0).to_dict())
print("Parcelas con llenado ≤ 7 días:", largo_llen[largo_llen <= 7].index.tolist())
print("Parcelas con vegetativa ≤ 7 días:", largo_veg[largo_veg <= 7].index.tolist())

Días de la etapa vegetativa: {'count': 197.0, 'mean': 61.0, 'std': 23.0, 'min': 13.0, '25%': 45.0, '50%': 61.0, '75%': 81.0, 'max': 110.0}
Días de la etapa de llenado: {'count': 197.0, 'mean': 37.0, 'std': 14.0, 'min': 16.0, '25%': 28.0, '50%': 33.0, '75%': 42.0, 'max': 111.0}
Parcelas con llenado ≤ 7 días: []
Parcelas con vegetativa ≤ 7 días: []


**Interpretaciones**

- **Etapa vegetativa**: de $13$ a $110$ días (mediana $61$).
- **Llenado de grano**: de $16$ a $111$ días (mediana $33$).
- Ninguna parcela tiene etapas de 7 días o menos.

#  Propiedades del suelo (SoilGrids 2.0, 0–30 cm)
# Consideramos que esta parte es muy importante ya que la base de datos dada no cuenta con ninguna propiedad del suelo
**Fuente**: SoilGrids 2.0 es un mapa global de propiedades del suelo a $250$ m, con seis profundidades estándar. Se generó con aprendizaje automático a partir de perfiles de suelo y covariables ambientales, e incluye la incertidumbre de cada predicción (Poggio et al., 2021). Se accede como activo de Earth Engine en `projects/soilgrids-isric`.

**Tenemos:**
1. **Variables**: arcilla, arena, carbono orgánico (SOC), densidad aparente y pH en agua. Se usa la media de la predicción (`_mean`).
2. **Capa de 0 a 30 cm** como **promedio ponderado por espesor** de las capas de 0–5, 5–15 y 15–30 cm (pesos $5$, $10$ y $15$).
3. **Conversión a unidades físicas**: SoilGrids almacena los valores como enteros multiplicados por 10 o 100 (Poggio et al., 2021). El notebook divide entre 10 la arcilla, la arena, el SOC y el pH, y entre 100 la densidad. Después de convertir, cada variable debe caer en un **rango físico plausible**, lo que detecta una conversión olvidada.
4. **Limo como complemento**: $100 - \text{arcilla} - \text{arena}$. Antes se verifica que arcilla + arena $\le 100$.
5. **Agua disponible**: $\theta_{33} - \theta_{1500}$ con las funciones de pedotransferencia de Saxton y Rawls (2006), a partir de la arena, la arcilla y la materia orgánica. Se multiplica por $300$ mm de profundidad.

**Justificación**
- La capa de 0 a 30 cm representa la parte superficial del perfil, donde se espera la mayor actividad de raíces de un cereal de temporal. Ponderar por espesor evita que la capa de 0–5 cm, la más delgada, pese lo mismo que las demás.
- Las ecuaciones de Saxton y Rawls (2006) estiman la retención de agua solo a partir de la textura y la materia orgánica, que son justamente las variables que ofrece SoilGrids. El código reproduce sus ecuaciones para $\theta_{1500}$ y $\theta_{33}$. 
- La materia orgánica se obtiene como $\text{SOC} \times 1.724$, el factor convencional que supone un 58 % de carbono en la materia orgánica. 

In [ ]:
#  SoilGrids 2.0 (0–30 cm) por parcela, con conversión de unidades y agua disponible.
ESCALA_SUELO_M: float = 50.0
CAPAS: dict[str, int] = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}     # espesor en cm
PROFUNDIDAD_MM: int = 300

# variable SoilGrids: (divisor, nombre final, rango físico plausible tras convertir)
VARIABLES_SUELO: dict[str, tuple[float, str, tuple[float, float]]] = {
    "clay":  (10.0,  "soil_arcilla_pct",    (0.0, 100.0)),
    "sand":  (10.0,  "soil_arena_pct",      (0.0, 100.0)),
    "soc":   (10.0,  "soil_soc_g_kg",       (0.0, 300.0)),
    "bdod":  (100.0, "soil_densidad_g_cm3", (0.8, 1.9)),
    "phh2o": (10.0,  "soil_ph",             (3.5, 9.5)),
}


def extraer_variable_suelo(var: str, divisor: float, nombre: str) -> pd.DataFrame:
    """Promedio 0–30 cm ponderado por espesor, ya en unidades físicas."""
    bandas = [f"{var}_{capa}_mean" for capa in CAPAS]
    imagen = ee.Image(f"projects/soilgrids-isric/{var}_mean").select(bandas)
    tabla = extraer_por_parcela(imagen, ESCALA_SUELO_M)
    faltan = set(bandas) - set(tabla.columns)
    if faltan:
        raise KeyError(f"{var}: Earth Engine no devolvió {faltan}")
    ponderado = sum(tabla[b] * esp for b, esp in zip(bandas, CAPAS.values())) / sum(CAPAS.values())
    return pd.DataFrame({CLAVE: tabla[CLAVE], nombre: ponderado / divisor})


partes = [extraer_variable_suelo(v, div, nom) for v, (div, nom, _) in VARIABLES_SUELO.items()]
suelo = partes[0]
for p in partes[1:]:
    suelo = suelo.merge(p, on=CLAVE, how="inner", validate="one_to_one")

# Contratos: 197 parcelas, sin NaN y en rango físico (detecta conversiones olvidadas).
assert len(suelo) == 197 and suelo[CLAVE].is_unique
assert suelo.drop(columns=CLAVE).notna().all().all(), "SoilGrids sin valor en alguna parcela"
for _, nombre, (minimo, maximo) in VARIABLES_SUELO.values():
    fuera = ~suelo[nombre].between(minimo, maximo)
    assert not fuera.any(), f"{nombre} fuera de [{minimo}, {maximo}]: {suelo.loc[fuera, nombre].tolist()[:5]}"
assert (suelo["soil_arcilla_pct"] + suelo["soil_arena_pct"] <= 100).all(), "Arcilla + arena > 100 %"

suelo["soil_limo_pct"] = 100 - suelo["soil_arcilla_pct"] - suelo["soil_arena_pct"]

# Agua disponible (m³/m³) = θ33 − θ1500, ecuaciones de Saxton y Rawls (2006).
def agua_disponible_saxton_rawls(arena_pct: pd.Series, arcilla_pct: pd.Series,
                                 soc_g_kg: pd.Series) :
    s, c = arena_pct / 100, arcilla_pct / 100
    mo = soc_g_kg / 10 * 1.724                       # materia orgánica (%), factor de van Bemmelen
    t1500 = (-0.024 * s + 0.487 * c + 0.006 * mo + 0.005 * s * mo
             - 0.013 * c * mo + 0.068 * s * c + 0.031)
    theta_1500 = t1500 + (0.14 * t1500 - 0.02)
    t33 = (-0.251 * s + 0.195 * c + 0.011 * mo + 0.006 * s * mo
           - 0.027 * c * mo + 0.452 * s * c + 0.299)
    theta_33 = t33 + (1.283 * t33 ** 2 - 0.374 * t33 - 0.015)
    return theta_33 - theta_1500


suelo["soil_agua_disp_mm"] = (
    agua_disponible_saxton_rawls(suelo["soil_arena_pct"], suelo["soil_arcilla_pct"],
                                 suelo["soil_soc_g_kg"]) * PROFUNDIDAD_MM
)
assert suelo["soil_agua_disp_mm"].between(5, 120).all(), "Agua disponible fuera de rango plausible"

print(suelo.drop(columns=CLAVE).describe().round(2).T[["min", "25%", "50%", "75%", "max"]])

# El suelo no depende del píxel de clima: debería variar mucho dentro de cada píxel.
print("\nVarianza dentro de los píxeles de clima:")
for col in ("soil_arcilla_pct", "soil_soc_g_kg", "soil_ph", "soil_agua_disp_mm"):
    print(f"  {col:22s} {fraccion_dentro_pixel(suelo.set_index(CLAVE)[col]):5.0%}")

suelo.to_parquet(DIR_INTERIM / "soil_parcela.parquet", index=False)
print(f"\nGuardado soil_parcela: {suelo.shape}")

                       min    25%    50%    75%    max
soil_arcilla_pct     20.43  25.28  26.61  28.88  31.82
soil_arena_pct       38.40  42.83  44.90  46.63  49.24
soil_soc_g_kg        13.90  17.23  19.49  21.41  27.87
soil_densidad_g_cm3   1.16   1.24   1.26   1.27   1.34
soil_ph               6.00   6.32   6.40   6.53   7.11
soil_limo_pct        24.19  27.29  28.56  29.83  32.47
soil_agua_disp_mm    36.18  38.17  39.21  39.70  41.70

Varianza dentro de los píxeles de clima:
  soil_arcilla_pct         65%
  soil_soc_g_kg            39%
  soil_ph                  38%
  soil_agua_disp_mm        32%

Guardado soil_parcela: (197, 8)


**Interpretaciones**

- **Rangos**: arcilla de $20$ a $32\%$, arena de $38$ a $49\%$, SOC de $13.9$ a $27.9$ g/kg, densidad de $1.16$ a $1.34$ g/cm³ y pH de $6.0$ a $7.1$. Todas las variables pasaron los contratos de rango y no hay faltantes.
- **Agua disponible**: de $36.2$ a $41.7$ mm, un rango muy estrecho. Con tan poca variación entre parcelas, su capacidad para distinguir rendimientos es limitada.
- **Variación dentro de los píxeles de clima**: de $32\%$ (agua disponible) a $65\%$ (arcilla). A diferencia de la lluvia, el suelo sí aporta variación propia de cada parcela.

**Limitaciones y supuestos pendientes**
- **Exactitud de SoilGrids**: en validación cruzada, la eficiencia del modelo va de $0.31$ a $0.74$ según la propiedad, y la arcilla se modela peor que las otras fracciones de textura (Poggio et al., 2021). Es importante recordar que los valores por parcela son predicciones, no mediciones.
- **Factor 1.724**: Pribyl (2010) revisa este factor y concluye que un valor cercano a $2$ suele ser más preciso. Con 1.724 la materia orgánica, y por lo tanto el agua disponible, podría estar subestimada. (necesitas investigar más)

In [ ]:
#· SoilGrids 2.0 (0–30 cm) por parcela, con conversión de unidades y agua disponible.
ESCALA_SUELO_M: float = 50.0
CAPAS: dict[str, int] = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}     # espesor en cm
PROFUNDIDAD_MM: int = 300

# variable SoilGrids: (divisor, nombre final, rango físico plausible tras convertir)
VARIABLES_SUELO: dict[str, tuple[float, str, tuple[float, float]]] = {
    "clay":  (10.0,  "soil_arcilla_pct",    (0.0, 100.0)),
    "sand":  (10.0,  "soil_arena_pct",      (0.0, 100.0)),
    "soc":   (10.0,  "soil_soc_g_kg",       (0.0, 300.0)),
    "bdod":  (100.0, "soil_densidad_g_cm3", (0.8, 1.9)),
    "phh2o": (10.0,  "soil_ph",             (3.5, 9.5)),
}

# Promedio 0–30 cm ponderado por espesor, ya en unidades físicas.
def extraer_variable_suelo(var: str, divisor: float, nombre: str) :
    bandas = [f"{var}_{capa}_mean" for capa in CAPAS]
    imagen = ee.Image(f"projects/soilgrids-isric/{var}_mean").select(bandas)
    tabla = extraer_por_parcela(imagen, ESCALA_SUELO_M)
    faltan = set(bandas) - set(tabla.columns)
    if faltan:
        raise KeyError(f"{var}: Earth Engine no devolvió {faltan}")
    ponderado = sum(tabla[b] * esp for b, esp in zip(bandas, CAPAS.values())) / sum(CAPAS.values())
    return pd.DataFrame({CLAVE: tabla[CLAVE], nombre: ponderado / divisor})


partes = [extraer_variable_suelo(v, div, nom) for v, (div, nom, _) in VARIABLES_SUELO.items()]
suelo = partes[0]
for p in partes[1:]:
    suelo = suelo.merge(p, on=CLAVE, how="inner", validate="one_to_one")

# Contratos: 197 parcelas, sin NaN y en rango físico (detecta conversiones olvidadas).
assert len(suelo) == 197 and suelo[CLAVE].is_unique
assert suelo.drop(columns=CLAVE).notna().all().all(), "SoilGrids sin valor en alguna parcela"
for _, nombre, (minimo, maximo) in VARIABLES_SUELO.values():
    fuera = ~suelo[nombre].between(minimo, maximo)
    assert not fuera.any(), f"{nombre} fuera de [{minimo}, {maximo}]: {suelo.loc[fuera, nombre].tolist()[:5]}"
assert (suelo["soil_arcilla_pct"] + suelo["soil_arena_pct"] <= 100).all(), "Arcilla + arena > 100 %"

suelo["soil_limo_pct"] = 100 - suelo["soil_arcilla_pct"] - suelo["soil_arena_pct"]

# Agua disponible (m³/m³) = θ33 − θ1500, ecuaciones de Saxton y Rawls (2006).
def agua_disponible_saxton_rawls(arena_pct: pd.Series, arcilla_pct: pd.Series,
                                 soc_g_kg: pd.Series):
    s, c = arena_pct / 100, arcilla_pct / 100
    mo = soc_g_kg / 10 * 1.724                       # materia orgánica (%), factor de van Bemmelen
    t1500 = (-0.024 * s + 0.487 * c + 0.006 * mo + 0.005 * s * mo
             - 0.013 * c * mo + 0.068 * s * c + 0.031)
    theta_1500 = t1500 + (0.14 * t1500 - 0.02)
    t33 = (-0.251 * s + 0.195 * c + 0.011 * mo + 0.006 * s * mo
           - 0.027 * c * mo + 0.452 * s * c + 0.299)
    theta_33 = t33 + (1.283 * t33 ** 2 - 0.374 * t33 - 0.015)
    return theta_33 - theta_1500


suelo["soil_agua_disp_mm"] = (
    agua_disponible_saxton_rawls(suelo["soil_arena_pct"], suelo["soil_arcilla_pct"],
                                 suelo["soil_soc_g_kg"]) * PROFUNDIDAD_MM
)
assert suelo["soil_agua_disp_mm"].between(5, 120).all(), "Agua disponible fuera de rango plausible"

print(suelo.drop(columns=CLAVE).describe().round(2).T[["min", "25%", "50%", "75%", "max"]])

# El suelo no depende del píxel de clima: debería variar mucho dentro de cada píxel.
print("\nVarianza dentro de los píxeles de clima:")
for col in ("soil_arcilla_pct", "soil_soc_g_kg", "soil_ph", "soil_agua_disp_mm"):
    print(f"  {col:22s} {fraccion_dentro_pixel(suelo.set_index(CLAVE)[col]):5.0%}")

suelo.to_parquet(DIR_INTERIM / "soil_parcela.parquet", index=False)
print(f"\nGuardado soil_parcela: {suelo.shape}")

                       min    25%    50%    75%    max
soil_arcilla_pct     20.43  25.28  26.61  28.88  31.82
soil_arena_pct       38.40  42.83  44.90  46.63  49.24
soil_soc_g_kg        13.90  17.23  19.49  21.41  27.87
soil_densidad_g_cm3   1.16   1.24   1.26   1.27   1.34
soil_ph               6.00   6.32   6.40   6.53   7.11
soil_limo_pct        24.19  27.29  28.56  29.83  32.47
soil_agua_disp_mm    36.18  38.17  39.21  39.70  41.70

Varianza dentro de los píxeles de clima:
  soil_arcilla_pct         65%
  soil_soc_g_kg            39%
  soil_ph                  38%
  soil_agua_disp_mm        32%

Guardado soil_parcela: (197, 8)



### Relación del suelo con la geografía

**Pregunta**: ¿el suelo aporta información independiente o repite lo que ya contienen la topografía y la ubicación? Se calculan las correlaciones de la arcilla, el SOC y el pH con la elevación media y la latitud, y sus medianas por municipio.


In [ ]:
# ¿El suelo sigue a la geografía o es independiente?
topo_full = pd.read_parquet(DIR_INTERIM / "topo_parcela.parquet", columns=[CLAVE, "elev_media_m"])
s = (suelo.merge(topo_full, on=CLAVE)
          .merge(parcelas[[CLAVE, "municipio"]], on=CLAVE)
          .merge(latitud, on=CLAVE))
print(s[["soil_arcilla_pct", "soil_soc_g_kg", "soil_ph", "elev_media_m", "lat"]]
      .corr().round(2).loc[["soil_arcilla_pct", "soil_soc_g_kg", "soil_ph"], ["elev_media_m", "lat"]])
print("\nMediana por municipio:")
print(s.groupby("municipio")[["soil_arcilla_pct", "soil_soc_g_kg", "soil_ph"]].median().round(1))

NameError: name 'suelo' is not defined

##  Meteorología diaria de ERA5-Land (2022–2025)

**Fuente**: ERA5-Land es un reanálisis de superficie terrestre del ECMWF con una resolución de $\approx 9$ km, obtenido al volver a ejecutar el componente terrestre de ERA5 con forzamiento atmosférico (Muñoz-Sabater et al., 2021). Se usa la colección de agregados diarios `ECMWF/ERA5_LAND/DAILY_AGGR`.

**Variables**: temperatura máxima y media a 2 m, temperatura de punto de rocío, humedad volumétrica del suelo en dos capas (0–7 y 7–28 cm) y radiación solar descendente acumulada.

**Decisiones**
1. **Misma estrategia de descarga que CHIRPS**: una llamada por mes, caché en disco, reintentos y contratos sobre el número de imágenes y de filas.
2. **Parcelas sobre píxeles de agua**: para `AGC_009` y `AGC_049`, cuyo píxel de ERA5-Land corresponde a un cuerpo de agua (laguna de Tecocomulco y laguna de Chignahuapan), se promedian los **píxeles terrestres dentro de un círculo de 15 km**. Se marcan con `diag_era5_vecino`.
3. **Versión de la caché** (`era5_diario_v2`): cuando cambia el método, la caché empieza en una carpeta nueva y vacía, para que nunca se mezclen datos de dos métodos.
4. **Conversión de unidades en el momento de la extracción**: kelvin a °C y J/m² a MJ/m².
5. **Déficit de presión de vapor (VPD)**: $e_s(T_{\text{media}}) - e_a(T_d)$, con la ecuación de Tetens de FAO-56 (ecuación 11; Allen et al., 1998), recortado a $\ge 0$.
6. **Contratos de rango físico**: Tmax entre $-10$ y $45\ °\text{C}$, humedad del suelo entre $0$ y $0.8$, radiación entre $0$ y $40$ MJ/m², y punto de rocío menor o igual que la temperatura media.

**Justificación**
- ERA5-Land solo modela superficie terrestre, por lo que sus píxeles sobre cuerpos de agua no tienen dato. Promediar los píxeles terrestres vecinos asigna a esas dos parcelas el clima de su entorno sin eliminarlas del análisis, y la bandera conserva la trazabilidad
- Versionar la caché garantiza que un resultado provenga de un único método de extracción.
- Los contratos de rango se diseñaron para detectar conversiones de unidades olvidadas, que son errores silenciosos frecuentes con datos de reanálisis. 

In [ ]:
# ERA5-Land diario 2022–2025 por parcela (Earth Engine → tabla larga con caché mensual).
COLECCION_ERA5: str = "ECMWF/ERA5_LAND/DAILY_AGGR"
ESCALA_ERA5_M: float = 1_000.0
# v2 = extracción con búfer terrestre para AGC_009 y AGC_049. Si cambia el método,
# se sube la versión: la carpeta nueva empieza vacía y nunca se mezclan datos viejos.
DIR_CACHE_ERA5: Path = DIR_INTERIM / "era5_diario_v2"
DIR_CACHE_ERA5.mkdir(parents=True, exist_ok=True)

BANDAS_ERA5: dict[str, str] = {
    "temperature_2m_max": "tmax_k",
    "temperature_2m": "tmedia_k",
    "dewpoint_temperature_2m": "td_k",
    "volumetric_soil_water_layer_1": "sm1",          # 0–7 cm, m³/m³
    "volumetric_soil_water_layer_2": "sm2",          # 7–28 cm, m³/m³
    "surface_solar_radiation_downwards_sum": "rs_j_m2",
}

# Parcelas cuyo píxel ERA5-Land es agua (Tecocomulco y laguna de Chignahuapan).
# Para ellas se promedian los píxeles TERRESTRES dentro de un círculo de 15 km.
PARCELAS_VECINO: list[str] = ["AGC_009", "AGC_049"]
RADIO_VECINO_M: float = 15_000.0
parcelas_vecino_ee = (parcelas_ee
                      .filter(ee.Filter.inList(CLAVE, PARCELAS_VECINO))
                      .map(lambda f: f.buffer(RADIO_VECINO_M)))
assert parcelas_vecino_ee.size().getInfo() == len(PARCELAS_VECINO)

# Descarga de ERA5-Land diario para un mes, con reintentos ante errores transitorios.
def descargar_era5_mes(periodo: pd.Period, reintentos: int = 3) :
    inicio = periodo.start_time.strftime("%Y-%m-%d")
    fin = (periodo + 1).start_time.strftime("%Y-%m-%d")
    coleccion = ee.ImageCollection(COLECCION_ERA5).filterDate(inicio, fin).select(list(BANDAS_ERA5))

    n_imagenes = coleccion.size().getInfo()
    if n_imagenes != periodo.days_in_month:
        raise ValueError(f"{periodo}: {n_imagenes} imágenes; se esperaban {periodo.days_in_month}")

    imagen_mes = coleccion.toBands()       
#   reduceRegions con reintentos ante errores transitorios del servidor.

    def _extraer(coleccion_fc: ee.FeatureCollection) :
            try:
                res = imagen_mes.reduceRegions(
                    collection=coleccion_fc, reducer=ee.Reducer.mean(),
                    scale=ESCALA_ERA5_M, tileScale=4,
                ).getInfo()
                return pd.DataFrame([f["properties"] for f in res["features"]])
            except ee.EEException as error:
                if intento == reintentos:
                    raise
                logger.warning("%s: intento %d falló (%s); reintentando…", periodo, intento, error)
                time.sleep(10 * intento)

    ancho = _extraer(parcelas_ee)                    # 197 parcelas con su propia geometría
    ancho_vecino = _extraer(parcelas_vecino_ee)      # 2 parcelas con búfer terrestre de 15 km
    ancho = pd.concat([ancho[~ancho[CLAVE].isin(PARCELAS_VECINO)], ancho_vecino], ignore_index=True)

    largo = ancho.melt(id_vars=CLAVE, var_name="banda", value_name="valor")
    largo["fecha"] = pd.to_datetime(largo["banda"].str[:8], format="%Y%m%d")
    largo["variable"] = largo["banda"].str[9:].map(BANDAS_ERA5)
    if largo["variable"].isna().any():
        raise ValueError(f"{periodo}: bandas no reconocidas {largo.loc[largo['variable'].isna(), 'banda'].unique()[:3]}")
    return (largo.pivot(index=[CLAVE, "fecha"], columns="variable", values="valor")
            .reset_index().rename_axis(columns=None))


inicio_reloj = time.perf_counter()
for periodo in MESES:
    ruta_mes = DIR_CACHE_ERA5 / f"{periodo.strftime('%Y_%m')}.parquet"
    if ruta_mes.exists():
        continue
    descargar_era5_mes(periodo).to_parquet(ruta_mes, index=False)
    logger.info("ERA5-Land %s descargado", periodo)
logger.info("Descarga terminada en %.1f min", (time.perf_counter() - inicio_reloj) / 60)
n_cache = len(list(DIR_CACHE_ERA5.glob("*.parquet")))
logger.info("Meses en caché: %d de %d (carpeta: %s)", n_cache, len(MESES), DIR_CACHE_ERA5.name)
era5 = (pd.concat([pd.read_parquet(r) for r in sorted(DIR_CACHE_ERA5.glob("*.parquet"))],
                  ignore_index=True)
        .sort_values([CLAVE, "fecha"]).reset_index(drop=True))

# Conversión de unidades (aquí, no al ensamblar).
era5["tmax_c"] = era5.pop("tmax_k") - 273.15
era5["tmedia_c"] = era5.pop("tmedia_k") - 273.15
era5["td_c"] = era5.pop("td_k") - 273.15
era5["rs_mj_m2"] = era5.pop("rs_j_m2") / 1e6


def presion_saturacion_kpa(t_c: pd.Series) -> pd.Series:
    """Presión de vapor a saturación (Tetens, FAO-56 ec. 11)."""
    return 0.6108 * np.exp(17.27 * t_c / (t_c + 237.3))


era5["vpd_kpa"] = (presion_saturacion_kpa(era5["tmedia_c"]) - presion_saturacion_kpa(era5["td_c"])).clip(lower=0)

# Contratos.
assert len(era5) == 197 * n_dias, f"Filas: {len(era5)} (esperadas {197 * n_dias})"
assert not era5.duplicated([CLAVE, "fecha"]).any()
assert era5.drop(columns=[CLAVE, "fecha"]).notna().all().all(), "ERA5 con días sin valor"
assert era5["tmax_c"].between(-10, 45).all(), "Tmax fuera de rango (¿falta convertir de K?)"
assert era5[["sm1", "sm2"]].stack().between(0, 0.8).all(), "Humedad del suelo fuera de [0, 0.8]"
assert era5["rs_mj_m2"].between(0, 40).all(), "Radiación fuera de rango (¿falta dividir entre 1e6?)"
assert (era5["td_c"] <= era5["tmedia_c"] + 0.5).all(), "Punto de rocío mayor que la temperatura"

era5.to_parquet(DIR_INTERIM / "era5_diario_parcela.parquet", index=False)
print(era5.drop(columns=[CLAVE, "fecha"]).describe().round(2).T[["min", "50%", "max"]])

INFO | ERA5-Land 2022-01 descargado
INFO | ERA5-Land 2022-02 descargado
INFO | ERA5-Land 2022-03 descargado
INFO | ERA5-Land 2022-04 descargado
INFO | ERA5-Land 2022-05 descargado
INFO | ERA5-Land 2022-06 descargado
INFO | ERA5-Land 2022-07 descargado
INFO | ERA5-Land 2022-08 descargado
INFO | ERA5-Land 2022-09 descargado
INFO | ERA5-Land 2022-10 descargado
INFO | ERA5-Land 2022-11 descargado
INFO | ERA5-Land 2022-12 descargado
INFO | ERA5-Land 2023-01 descargado
INFO | ERA5-Land 2023-02 descargado
INFO | ERA5-Land 2023-03 descargado
INFO | ERA5-Land 2023-04 descargado
INFO | ERA5-Land 2023-05 descargado
INFO | ERA5-Land 2023-06 descargado
INFO | ERA5-Land 2023-07 descargado
INFO | ERA5-Land 2023-08 descargado
INFO | ERA5-Land 2023-09 descargado
INFO | ERA5-Land 2023-10 descargado
INFO | ERA5-Land 2023-11 descargado
INFO | ERA5-Land 2023-12 descargado
INFO | ERA5-Land 2024-01 descargado
INFO | ERA5-Land 2024-02 descargado
INFO | ERA5-Land 2024-03 descargado
INFO | ERA5-Land 2024-04 des

            min    50%    max
sm1        0.05   0.26   0.43
sm2        0.15   0.25   0.43
tmax_c     7.79  19.32  29.85
tmedia_c   4.78  13.30  22.50
td_c     -14.08   7.93  14.15
rs_mj_m2   2.92  20.90  31.86
vpd_kpa    0.01   0.39   1.94


Esta respuesta me da los siguientes resultados:
- **Rangos**: Tmax de $7.8$ a $29.9\ °\text{C}$ (mediana $19.3$), humedad del suelo de la capa 2 de $0.15$ a $0.43$ m³/m³, radiación de $2.9$ a $31.9$ MJ/m² y VPD de $0.01$ a $1.94$ kPa (mediana $0.39$)

**Haciendo una comparación empirica observamos que**
- La resolución de $\approx 9$ km de ERA5-Land es más gruesa que la malla de CHIRPS.
- FAO-56 recomienda calcular $e_s$ como el promedio de $e_s(T_{\max})$ y $e_s(T_{\min})$, porque la relación con la temperatura no es lineal. Calcularlo con la temperatura media lo subestima y, con él, el VPD (Allen et al., 1998). [Supuesto pendiente]

###  Creación de los (diag) Diagnóstico de valores vacíos en ERA5-Land

**Objetivo principal**: es distinguir entre faltantes **temporales**, como fallas de una fecha en el servidor, y faltantes **espaciales**,ejemplo cuando una parcela no tiene dato ningún día porque su píxel está enmascarado, son esos casos los que busca interpretar. Si una parcela está vacía todos los días, el problema es de ubicación.

In [ ]:
# ¿Qué variables, parcelas y fechas vienen vacías?
cols = era5.columns.drop([CLAVE, "fecha"])
vacio = era5[cols].isna()
print("NaN por variable:\n", vacio.sum()[vacio.sum() > 0].to_string(), "\n")

filas = era5[vacio.any(axis=1)]
print(f"Parcelas afectadas: {filas[CLAVE].nunique()} → {filas[CLAVE].value_counts().head(10).to_dict()}")
print(f"Fechas afectadas: {filas['fecha'].nunique()} de {era5['fecha'].nunique()}")
print("Por mes:", filas["fecha"].dt.to_period("M").value_counts().sort_index().to_dict(), "\n")

# ¿Hay parcelas vacías en TODOS los días? → píxel enmascarado, no fallo temporal.
siempre_vacia = vacio.groupby(era5[CLAVE]).all()
siempre_vacia = siempre_vacia[siempre_vacia.any(axis=1)]
print("Parcelas sin dato nunca:", siempre_vacia.index.tolist())
print(parcelas.loc[parcelas[CLAVE].isin(siempre_vacia.index), [CLAVE, "municipio"]].to_string(index=False))

NaN por variable:
 Series([], ) 

Parcelas afectadas: 0 → {}
Fechas afectadas: 0 de 1461
Por mes: {} 

Parcelas sin dato nunca: []
Empty DataFrame
Columns: [id_poligono, municipio]
Index: []


**Interpretaciones**
- Con la versión actual la v2 no queda **ningún valor vacío**: $0$ parcelas afectadas y $0$ de $1{,}461$ fechas 

### v2 Verificación del cambio de método (v1 a v2)

se busca confirmar que unicamente se modificó a las dos parcelas de agua y que el resto quedaron intactas. También se comprueba que los valores rellenados sean creibles o comprobables  frente al resto de las parcelas.

**Procedimiento**: se guarda una copia de la versión anterior (`era5_anterior`), se ejecuta la versión 2 y se comparan las dos tablas. Al final se borra la caché mensual.


In [ ]:
era5_anterior = era5.copy()

In [ ]:
PARCELAS_VECINO: list[str] = ["AGC_009", "AGC_049"]
cols = era5.columns.drop([CLAVE, "fecha"])
sin_cambio = lambda d: d[~d[CLAVE].isin(PARCELAS_VECINO)].set_index([CLAVE, "fecha"])[cols]
print("Cambio máximo en las otras 195 parcelas:")
print((sin_cambio(era5_anterior) - sin_cambio(era5)).abs().max().round(6).to_string())

print("\nValores rellenados vs. mediana del resto (julio 2025):")
jul = era5[era5["fecha"].between("2025-07-01", "2025-07-31")]
print(jul.groupby(jul[CLAVE].isin(PARCELAS_VECINO))[["tmax_c", "sm2", "rs_mj_m2"]].median().round(2))

Cambio máximo en las otras 195 parcelas:
sm1         0.0
sm2         0.0
tmax_c      0.0
tmedia_c    0.0
td_c        0.0
rs_mj_m2    0.0
vpd_kpa     0.0

Valores rellenados vs. mediana del resto (julio 2025):
             tmax_c   sm2  rs_mj_m2
id_poligono                        
False         18.50  0.39     21.41
True          18.21  0.40     21.09


**Interpretaciones importantes**

- **Parcelas sin cambio**: el cambio máximo en las otras parcelas es $0$ en todas las variables. La corrección solo afectó a `AGC_009` y `AGC_049`
- **Valores rellenados**: en julio de 2025 son verosímiles frente a la mediana del resto: Tmax de $18.21$ frente a $18.50\ °\text{C}$, humedad de la capa 2 de $0.40$ frente a $0.39$ y radiación de $21.09$ frente a $21.41$ MJ/m².


In [52]:
for ruta in DIR_CACHE_ERA5.glob("*.parquet"):
    ruta.unlink()
print("Archivos restantes en caché:", len(list(DIR_CACHE_ERA5.glob("*.parquet"))))

Archivos restantes en caché: 0



## Variables de ERA5-Land por etapa fenológica

**QUE SE HIZO:**
1. **Corrección delta de la Tmax**: para cada parcela y mes se calcula

$$ \Delta = T^{\text{mensual}}_{\max,\ \text{corregida}} - \overline{T}^{\ \text{ERA5}}_{\max} $$

  y se suma a cada día. ERA5 aporta la **variación diaria** y la Tmax corregida por elevación lo que aporta el **nivel**. 
  
  * La columna se elimina antes de volver a crearla, para que la celda pueda ejecutarse varias veces sin duplicarla.

2. **FAPAR diario** a partir de la curva suavizada de NDVI de Planet, con la relación lineal $\text{FAPAR} = 1.1638\,\text{NDVI} - 0.1426$ (Myneni y Williams, 1994), recortada a $[0,\ 0.95]$

3. **APAR**: $\text{FAPAR} \times 0.48 \times R_s$, la radiación fotosintéticamente activa absorbida. Se acumula en el ciclo y desde el pico 

4. **Variables por etapa**: días con Tmax por encima de $28$ y $30\ °\text{C}$ en floración y llenado, Tmax media en floración, VPD en floración y llenado, humedad de la capa 1 en `sos20` $\pm 7$ días, humedad de la capa 2 en floración y llenado (media y mínimo) y radiación del ciclo.

**Justificación**
- La corrección delta, o escalamiento lineal, ajusta el nivel medio de una variable modelada a una referencia sin cambiar su variación diaria (Teutschbein y Seibert, 2013). Las celdas de ERA5-Land, de $\approx 9$ km, promedian alturas distintas, mientras que la Tmax del Paso 3 está corregida por la elevación de cada parcela.
- El modelo de eficiencia en el uso de la radiación relaciona la producción de biomasa con la radiación que absorbe el cultivo (Monteith, 1972). El APAR traslada esa idea a cada parcela combinando la radiación de ERA5-Land con el verdor observado por Planet.
- Los umbrales de $28$ y $30\ °\text{C}$ para estrés por calor son **criterios prácticos**.


In [ ]:
#Variables de ERA5-Land en las etapas fenológicas de cada parcela (prefijo era5_).
UMBRALES_CALOR_C: tuple[float, ...] = (28.0, 30.0)
DIAS_SIEMBRA_VENTANA: int = 7          # humedad del suelo en sos20 ± 7 días

# Corrección de nivel de la Tmax (método delta contra la Tmax mensual corregida por elevación).
era5["anio"], era5["mes"] = era5["fecha"].dt.year, era5["fecha"].dt.month
media_era5 = era5.groupby([CLAVE, "anio", "mes"], as_index=False)["tmax_c"].mean()
delta = (media_era5.merge(clima[[CLAVE, "anio", "mes", "tmax_c_ajust"]],
                          on=[CLAVE, "anio", "mes"], validate="one_to_one")
         .assign(delta_tmax=lambda d: d["tmax_c_ajust"] - d["tmax_c"])
         [[CLAVE, "anio", "mes", "delta_tmax"]])
# Re-ejecutable: se eliminan las columnas que esta celda crea antes de volver a crearlas.
era5 = (era5.drop(columns=["delta_tmax", "tmax_c_corr"], errors="ignore")
            .merge(delta, on=[CLAVE, "anio", "mes"], how="left", validate="many_to_one"))
era5["tmax_c_corr"] = era5["tmax_c"] + era5["delta_tmax"]
print("Corrección delta de Tmax (°C):", era5["delta_tmax"].describe().round(2)[["min", "50%", "max"]].to_dict())

# FAPAR diario a partir de la curva suavizada de NDVI de Planet (2025, mar–nov).
curvas = pd.read_parquet(DIR_INTERIM / "curvas_diarias_pl.parquet", columns=[CLAVE, "fecha", "pl_ndvi_ws"])
curvas["fecha"] = curvas["fecha"].astype("datetime64[ns]")
era5["fecha"] = era5["fecha"].astype("datetime64[ns]")      # misma resolución en ambas llaves# Relación lineal NDVI–FAPAR de Myneni y Williams (1994): aproximación para comparar parcelas.
curvas["fapar"] = (1.1638 * curvas["pl_ndvi_ws"] - 0.1426).clip(0, 0.95)

e25 = (era5[era5["anio"] == ANIO_CICLO]
       .assign(doy=lambda d: d["fecha"].dt.dayofyear)
       .merge(curvas[[CLAVE, "fecha", "fapar"]], on=[CLAVE, "fecha"], how="left", validate="one_to_one"))

# Variables de ERA5-Land en las etapas fenológicas de cada parcela (prefijo era5_).
def variables_era5(g: pd.DataFrame, f: pd.Series) :
    sos20, centro, eos50, eos20 = (int(f[c]) for c in
                                   ("pl_sos20_doy", "pl_centro_pico_doy", "pl_eos50_doy", "pl_eos20_doy"))
    flor = g[g["doy"].between(centro - MEDIA_VENTANA_FLOR, centro + MEDIA_VENTANA_FLOR)]
    llen = g[g["doy"].between(centro + MEDIA_VENTANA_FLOR + 1, eos50)]
    siembra = g[g["doy"].between(sos20 - DIAS_SIEMBRA_VENTANA, sos20 + DIAS_SIEMBRA_VENTANA)]
    ciclo = g[g["doy"].between(sos20, eos20)]
    if ciclo["fapar"].isna().any():
        raise ValueError(f"{g[CLAVE].iloc[0]}: días del ciclo sin FAPAR")

    fila: dict[str, float] = {}
    for u in UMBRALES_CALOR_C:
        fila[f"era5_dias_tmax{int(u)}_flor"] = int((flor["tmax_c_corr"] > u).sum())
        fila[f"era5_dias_tmax{int(u)}_llen"] = int((llen["tmax_c_corr"] > u).sum())
    apar = ciclo["fapar"] * 0.48 * ciclo["rs_mj_m2"]
    fila |= {
        "era5_tmax_media_flor_c": float(flor["tmax_c_corr"].mean()),
        "era5_vpd_medio_flor_kpa": float(flor["vpd_kpa"].mean()),
        "era5_vpd_medio_llen_kpa": float(llen["vpd_kpa"].mean()),
        "era5_sm1_siembra": float(siembra["sm1"].mean()),
        "era5_sm2_flor": float(flor["sm2"].mean()),
        "era5_sm2_llen": float(llen["sm2"].mean()),
        "era5_sm2_min_llen": float(llen["sm2"].min()),
        "era5_rs_ciclo_mj_m2": float(ciclo["rs_mj_m2"].sum()),
        "era5_apar_ciclo_mj_m2": float(apar.sum()),
        "era5_apar_post_pico_mj_m2": float(apar[ciclo["doy"] >= centro].sum()),   # llenado de grano
    }
    return fila


era5_parcela = pd.DataFrame([
    {CLAVE: pid, **variables_era5(g, fen.loc[pid])}
    for pid, g in e25.groupby(CLAVE, sort=True)
])

# Contratos.
# Parcelas cuyo píxel ERA5-Land es agua: su clima viene de los píxeles terrestres vecinos.
era5_parcela["diag_era5_vecino"] = era5_parcela[CLAVE].isin(PARCELAS_VECINO)

# Contratos.
assert len(era5_parcela) == 197 and era5_parcela[CLAVE].is_unique
assert int(era5_parcela["diag_era5_vecino"].sum()) == 2
assert era5_parcela.drop(columns=CLAVE).notna().all().all(), "Variables ERA5 con NaN"
assert (era5_parcela["era5_apar_ciclo_mj_m2"] > 0).all()

print(era5_parcela.drop(columns=CLAVE).describe().round(2).T[["min", "25%", "50%", "75%", "max"]])
print("\nVarianza dentro de los píxeles de clima:")
for col in [c for c in era5_parcela.columns if c != CLAVE and not c.startswith("diag_")]:
    print(f"  {col:28s} {fraccion_dentro_pixel(era5_parcela.set_index(CLAVE)[col]):5.0%}")

era5_parcela.to_parquet(DIR_INTERIM / "era5_parcela.parquet", index=False)
print(f"\nGuardado era5_parcela: {era5_parcela.shape}")

Corrección delta de Tmax (°C): {'min': 0.68, '50%': 4.16, 'max': 5.75}
                               min      25%      50%      75%      max
era5_dias_tmax28_flor         0.00     0.00     0.00     0.00     0.00
era5_dias_tmax28_llen         0.00     0.00     0.00     0.00     0.00
era5_dias_tmax30_flor         0.00     0.00     0.00     0.00     0.00
era5_dias_tmax30_llen         0.00     0.00     0.00     0.00     0.00
era5_tmax_media_flor_c       21.16    22.39    22.99    23.29    23.92
era5_vpd_medio_flor_kpa       0.16     0.22     0.25     0.30     0.40
era5_vpd_medio_llen_kpa       0.19     0.23     0.26     0.32     0.44
era5_sm1_siembra              0.28     0.35     0.36     0.39     0.42
era5_sm2_flor                 0.29     0.37     0.39     0.40     0.41
era5_sm2_llen                 0.27     0.35     0.37     0.38     0.40
era5_sm2_min_llen             0.25     0.28     0.30     0.34     0.36
era5_rs_ciclo_mj_m2        1707.76  2792.63  3000.96  3155.16  3456.78
era5_a

# **CONCLUSIONES Y INTERPRETACIONES**

- **Magnitud de la corrección delta**: entre $+0.68$ y $+5.75\ °\text{C}$ (mediana $+4.16\ °\text{C}$). La Tmax de ERA5-Land es sistemáticamente más fría que la Tmax corregida
- **Estrés por calor**: los cuatro conteos de días con Tmax mayor a $28$ o $30\ °\text{C}$ valen $0$ en todas las parcelas, porque la Tmax media en floración no pasa de $23.9\ °\text{C}$ son constantes.
- **APAR del ciclo**: la mediana es de $755$ MJ/m², de los cuales $343$ se acumulan después del pico.
- **Humedad del suelo**: la capa 2 en floración tiene una mediana de $0.39$ m³/m³ y la mínima del llenado, de $0.30$.
- **Variación dentro de los píxeles de clima**: va de $9\%$ (VPD en floración) a $86\%$ (radiación del ciclo). Como ERA5-Land tiene una malla más gruesa que la de clima, esta variación proviene sobre todo de las ventanas fenológicas, de la corrección delta y del FAPAR de cada parcela.
- **Productos**: `era5_diario_parcela.parquet` y `era5_parcela.parquet` ($197 \times 16$: clave, 14 variables y `diag_era5_vecino`).

**Limitaciones y supuestos pendientes para condsiderar en el EDA**
- **Corrección de $\approx 4\ °\text{C}$**: es grande. Conviene contrastar ambas fuentes con estaciones del Servicio Meteorológico Nacional para decidir cuál representa mejor el nivel real. Además, los métodos simples como el escalamiento lineal corrigen la media pero no la distribución, y su desempeño fuera del periodo de ajuste es cuestionable (Teutschbein y Seibert, 2013).
- **Inconsistencia de temperaturas**: el VPD usa la temperatura media de ERA5-Land **sin corregir**, mientras que la Tmax sí se corrige.
- **FAPAR**: la relación de Myneni y Williams (1994) es una aproximación general. Sus coeficientes dependen del sensor y del tipo de vegetación, así que el APAR sirve para comparar parcelas, no como valor absoluto.